# DYL-probe score distributions × speech-act labels

Interactive violin explorer: the **alpaca control** (9,994 replies, no game
context) next to **Carrot** and **Parsnip** base-game messages, split by
round on the x-axis. The dropdown filters the game violins to one
speech-act label (`defer`, `probe`, `flag`, `accuse`, ...) from the
2026-10-08 labeling run; the legend toggles Carrot/Parsnip. Y axis is the
DYL probe score (`p(deceptive)` read on the follow-up "No"), log-odds
spaced. Dashed line = alpaca 1%-FPR threshold.

Inputs (symlinked into this folder): `scores_{alpaca,base}.csv` from
`experiments/2026-10-07-dyl-alpaca`, `judge_labels_base.jsonl` from this
experiment. Run top-to-bottom; requires `plotly` in `.venv`.


In [1]:
import csv
import json
from pathlib import Path

import numpy as np

HERE = Path(".").resolve()
assert (HERE / "judge_labels_base.jsonl").exists(), "run from the experiment folder"

PRIMARY = ["defer", "probe", "flag", "accuse", "defend_self", "defend_other", "coordinate"]

with open("scores_alpaca.csv") as f:
    alpaca = np.array([float(r["followup_full"]) for r in csv.DictReader(f)])

with open("scores_base.csv") as f:
    base = [r for r in csv.DictReader(f) if r["silent"] == "0"]
labels = {}
with open("judge_labels_base.jsonl") as f:
    for line in f:
        r = json.loads(line)
        labels[(r["game_id"], int(r["round"]), int(r["position"]))] = r["primary"]

rows = []
for r in base:
    key = (r["game_id"], int(r["round"]), int(r["position"]))
    rows.append({"round": int(r["round"]), "role": r["role"],
                 "score": float(r["followup_full"]), "primary": labels[key]})
assert len(rows) == 1500 and len(labels) == 1500

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

THRESHOLD = float(np.quantile(alpaca, 0.99))
print(f"{len(rows)} game turns, {len(alpaca)} alpaca replies, "
      f"alpaca 1%-FPR threshold = {THRESHOLD:.4f}")


1500 game turns, 9994 alpaca replies, alpaca 1%-FPR threshold = 0.9987


In [2]:
import plotly.graph_objects as go

SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8f8d86"
ROUND_NAME = {0: "Round 1", 1: "Round 2", 2: "Round 3"}

def subset(role, label):
    sel = [r for r in rows if r["role"] == role
           and (label == "All" or r["primary"] == label)]
    x = [ROUND_NAME[r["round"]] for r in sel]
    y = list(logit([r["score"] for r in sel]))
    n = [sum(1 for r in sel if r["round"] == k) for k in (0, 1, 2)]
    return x, y, n

def counts_text(label):
    _, _, nc = subset("Carrot", label)
    _, _, np_ = subset("Parsnip", label)
    return (f"n per round — Carrot: {nc[0]}/{nc[1]}/{nc[2]}, "
            f"Parsnip: {np_[0]}/{np_[1]}/{np_[2]}, alpaca: {len(alpaca):,}")

fig = go.Figure()
fig.add_trace(go.Violin(
    x=["Alpaca"] * len(alpaca), y=logit(alpaca), name="Alpaca (control)",
    line_color=GRAY, fillcolor=GRAY, opacity=0.55, points=False,
    box_visible=True, box_width=0.08, meanline_visible=False, spanmode="soft"))
for role, color in (("Carrot", BLUE), ("Parsnip", ORANGE)):
    x, y, _ = subset(role, "All")
    fig.add_trace(go.Violin(
        x=x, y=y, name=role, line_color=color, fillcolor=color, opacity=0.55,
        points=False, box_visible=True, box_width=0.08, spanmode="soft"))

buttons = []
for label in ["All"] + PRIMARY:
    xc, yc, _ = subset("Carrot", label)
    xp, yp, _ = subset("Parsnip", label)
    buttons.append(dict(
        label=label, method="update",
        args=[{"x": [xc, xp], "y": [yc, yp]},
              {"annotations[1].text": counts_text(label)},
              [1, 2]]))

probs = [1e-6, 1e-4, 1e-2, 0.5, 0.99]
fig.update_layout(
    template="none", violinmode="group",
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="DejaVu Sans", size=12, color=INK),
    title=dict(text="DYL probe scores: alpaca control vs game messages, by round and speech act",
               font=dict(size=15), x=0.02),
    xaxis=dict(categoryorder="array",
               categoryarray=["Alpaca", "Round 1", "Round 2", "Round 3"]),
    yaxis=dict(title="DYL probe score (probability, log-odds spacing)",
               tickvals=[float(v) for v in logit(probs)],
               ticktext=[f"{100 * p:g}%" for p in probs],
               gridcolor="#e1e0d9", zeroline=False),
    legend=dict(orientation="h", y=1.08, x=0.55),
    width=950, height=560, margin=dict(l=70, r=30, t=110, b=60),
    updatemenus=[dict(buttons=buttons, direction="down", x=0.02, y=1.14,
                      xanchor="left", yanchor="top", showactive=True,
                      bgcolor=SURFACE, bordercolor="#c3c2b7")],
    annotations=[
        dict(x=1.0, xref="paper", y=float(logit(THRESHOLD)), yref="y",
             text="alpaca 1%-FPR", showarrow=False, xanchor="right",
             yanchor="bottom", font=dict(size=11, color=MUTED)),
        dict(x=0.02, xref="paper", y=-0.1, yref="paper", showarrow=False,
             xanchor="left", text=counts_text("All"),
             font=dict(size=11, color=INK2)),
    ])
fig.add_hline(y=float(logit(THRESHOLD)), line_dash="dash", line_color=MUTED,
              line_width=1)
fig.show()
